Step 2. Setup the βVAE model and define helper functions

In [ ]:
class TimeSeriesVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=15):
        super(TimeSeriesVAE, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2)
        )
        
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, input_dim),
            nn.Sigmoid()
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def encode(self, x):
        enc = self.encoder(x)
        mu = self.fc_mu(enc)
        logvar = self.fc_logvar(enc)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z, mu, logvar = self.encode(x)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=0.5):
    BCE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + beta * KLD

In [ ]:
def train_vae(model, dataloader, epochs=100, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for data in dataloader:
            data = data[0].to(device)
            optimizer.zero_grad()
            
            recon_batch, mu, logvar = model(data)
            loss = vae_loss(recon_batch, data, mu, logvar)
            
            loss.backward()
            train_loss += loss.item()
            optimizer.step()
        
        avg_loss = train_loss / len(dataloader.dataset)
        if (epoch+1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')
    
    return model

In [ ]:
def save_model(model, adata, model_path='sc_vae_model.pth'):
    
    torch.save({
        'model_state_dict': model.state_dict(),
        'input_dim': model.encoder[0].in_features, 
        'latent_dim': model.fc_mu.out_features
        }, model_path)
    
def load_model(model_path='sc_vae_model.pth', device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    checkpoint = torch.load(model_path, map_location=device)
    
    model = TimeSeriesVAE(
        input_dim=checkpoint['input_dim'],
        latent_dim=checkpoint['latent_dim']
    )
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    print(f"loading model from {model_path}")
    print(f"input dim: {checkpoint['input_dim']}, latent dim: {checkpoint['latent_dim']}")
    
    return model

In [ ]:
def reduce_total_expression(matrix, target_sum=10000):
    matrix = matrix.T
    result_matrix = np.zeros_like(matrix)
    
    for j in range(matrix.shape[1]): 
        cell_data = matrix[:, j]
        
        sorted_indices = np.argsort(cell_data)[::-1]
        sorted_values = cell_data[sorted_indices]
        
        cumulative_sum = np.cumsum(sorted_values)
        k = np.argmax(cumulative_sum >= target_sum) + 1
        
        if k == 0 or cumulative_sum[k-1] < target_sum:
            k = len(sorted_values)
            
        top_indices = sorted_indices[:k]
        top_values = cell_data[top_indices]
        
        scale_factor = target_sum / np.sum(top_values)
        scaled_values = top_values * scale_factor
        
        result_matrix[top_indices, j] = scaled_values
    
    return result_matrix.T

In [ ]:
def preprocess_cell_orders(adata):
    X = adata.X
    if hasattr(X, 'toarray'):
        X = X.toarray()
    cell_orders = []
    for cell in X:
        sorted_idx = np.argsort(cell)[::-1]
        sorted_vals = cell[sorted_idx]
        cumsum = np.cumsum(sorted_vals)
        cell_orders.append((sorted_idx, cumsum))
    return cell_orders

def median_dropout_for_X(cell_orders, X, target_umi=10000):
    dropout_rates = []
    for sorted_idx, cumsum_orig in cell_orders:
        total_orig = cumsum_orig[-1]
        if total_orig == 0:
            dropout_rates.append(1.0)
            continue
        threshold_orig = target_umi * total_orig / X 
        k = bisect_left(cumsum_orig, threshold_orig) + 1
        if k > len(cumsum_orig):
            k = len(cumsum_orig)
        n_genes = len(cumsum_orig)
        dropout_rates.append(1.0 - k / n_genes)
    return np.median(dropout_rates)